# Preprocessing Notebook
This notebook handles only data cleaning and saves the cleaned dataset for modeling.

In [ ]:
import pandas as pd
import numpy as np
import nltk
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')


In [ ]:
# Load dataset
df = pd.read_csv('fake_job_postings.csv')
df = df[['title', 'description', 'requirements', 'fraudulent']].copy()
df.dropna(subset=['description', 'requirements'], inplace=True)


In [ ]:
# Function to clean and lemmatize text
def clean_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML
    text = re.sub(r"[^a-zA-Z]", " ", text)  # Remove special characters
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    tokens = [t for t in tokens if t not in stop_words]

    lemmatizer = WordNetLemmatizer()
    pos_tags = pos_tag(tokens)
    
    def get_wordnet_pos(treebank_tag):
        if treebank_tag.startswith('J'):
            return 'a'
        elif treebank_tag.startswith('V'):
            return 'v'
        elif treebank_tag.startswith('N'):
            return 'n'
        elif treebank_tag.startswith('R'):
            return 'r'
        else:
            return 'n'

    tokens = [lemmatizer.lemmatize(t, get_wordnet_pos(p)) for t, p in pos_tags]
    return " ".join(tokens)


In [ ]:
# Apply cleaning
df['clean_description'] = df['description'].apply(clean_text)
df['clean_requirements'] = df['requirements'].apply(clean_text)

# Combine both fields
df['combined_text'] = df['clean_description'] + " " + df['clean_requirements']

# Final cleaned dataframe
clean_df = df[['title', 'combined_text', 'fraudulent']].copy()


In [ ]:
# Save to new CSV
clean_df.to_csv('cleaned_fake_job_postings.csv', index=False)
print("✅ Saved cleaned data to 'cleaned_fake_job_postings.csv'")
